In [1]:
import pandas as pd

In [9]:
# Load joined attribute table
df = pd.read_csv("../data/roi_pl.csv")


In [10]:
# Convert imagery date to datetime
df["date_start"] = pd.to_datetime(
   df["date_start"],
   dayfirst=True,
   errors="coerce",
)


In [11]:
# Count records for each protected landscape and imagery date
landscape_dates = (
   df.groupby(["NAME", "date_start"])
   .size()
   .reset_index(name="n_records")
)
# Order by protected landscape, then imagery date
landscape_dates = landscape_dates.sort_values(
   ["NAME", "date_start"]
).reset_index(drop=True)
landscape_dates

,NAME,date_start,n_records
0,Arnside & Silverdale,2018-06-22,6
1,Arnside & Silverdale,2018-10-18,14
2,Arnside & Silverdale,2019-04-22,3
3,Arnside & Silverdale,2019-05-14,8
4,Arnside & Silverdale,2019-05-15,84
...,...,...,...
1314,YORKSHIRE DALES,2025-07-12,80
1315,YORKSHIRE DALES,2025-07-13,137
1316,YORKSHIRE DALES,2025-08-12,465
1317,YORKSHIRE DALES,2025-08-13,8


In [12]:
landscape_dates_b = (
   df.groupby(["NAME", "date_start"])
   .size()
   .reset_index(name="n_records")
)
landscape_order = (
   landscape_dates.groupby("NAME")["date_start"]
   .min()
   .sort_values()
   .index
)
landscape_dates_b["NAME"] = pd.Categorical(
   landscape_dates_b["NAME"],
   categories=landscape_order,
   ordered=True,
)
landscape_dates_b = (
   landscape_dates_b
   .sort_values(["NAME", "date_start"])
   .reset_index(drop=True)
)
landscape_dates_b

,NAME,date_start,n_records
0,Surrey Hills,2000-10-10,1
1,Surrey Hills,2004-09-09,31
2,Surrey Hills,2005-07-17,5
3,Surrey Hills,2005-08-21,3
4,Surrey Hills,2005-11-09,16
...,...,...,...
1314,Arnside & Silverdale,2022-04-20,51
1315,Arnside & Silverdale,2022-06-23,3
1316,Arnside & Silverdale,2022-08-10,41
1317,Arnside & Silverdale,2025-04-30,3


In [13]:
landscape_dates.to_csv("../data/roi_pl_dates.csv", index=False)

In [14]:
landscape_dates_b.to_csv("../data/roi_pl_dates_b.csv", index=False)

In [16]:

df["date_start"] = pd.to_datetime(

    df["date_start"],

    dayfirst=True,

    errors="coerce",

)

# Keep only the latest imagery record for each tile

latest_tiles = (

    df.sort_values("date_start")

    .drop_duplicates(

        subset="tile_name",

        keep="last",

    )

    .copy()

)

# Add month and year for ordering

latest_tiles["month"] = latest_tiles["date_start"].dt.month

latest_tiles["year"] = latest_tiles["date_start"].dt.year

# Order by month, then year

latest_tiles = (

    latest_tiles

    .sort_values(

        ["month", "year", "tile_name"]

    )

    .reset_index(drop=True)

)

# Keep only the useful columns

latest_tiles = latest_tiles[

    [

        "tile_name",

        "NAME",

        "date_start",

        "month",

        "year",

    ]

]

latest_tiles


,tile_name,NAME,date_start,month,year
0,SU2257,North Wessex Downs,2023-03-09,3.0,2023.0
1,SX1498,Cornwall,2015-04-09,4.0,2015.0
2,NY6050,North Pennines,2021-04-19,4.0,2021.0
3,NY6051,North Pennines,2021-04-19,4.0,2021.0
4,NY6052,North Pennines,2021-04-19,4.0,2021.0
...,...,...,...,...,...
37435,SV9712,Isles Of Scilly,NaT,NaN,NaN
37436,SV9713,Isles Of Scilly,NaT,NaN,NaN
37437,SV9714,Isles Of Scilly,NaT,NaN,NaN
37438,SV9715,Isles Of Scilly,NaT,NaN,NaN


In [17]:
latest_tiles.to_csv("../data/roi_latest_tiles.csv", index=False)

In [20]:
df["date_start"] = pd.to_datetime(
   df["date_start"],
   dayfirst=True,
   errors="coerce",
)
# One row per tile
tile_dates = (
   df.groupby("tile_name")
   .agg(
       NAME=("NAME", "first"),
       n_dates=("date_start", "nunique"),
       available_dates=(
           "date_start",
           lambda x: ", ".join(
               d.strftime("%d/%m/%Y")
               for d in sorted(x.dropna().unique())
           )
       ),
   )
   .reset_index()
)
tile_dates

,tile_name,NAME,n_dates,available_dates
0,NT6906,NORTHUMBERLAND,2,"24/04/2020, 02/06/2023"
1,NT7005,NORTHUMBERLAND,2,"24/04/2020, 02/06/2023"
2,NT7006,NORTHUMBERLAND,2,"24/04/2020, 02/06/2023"
3,NT7104,NORTHUMBERLAND,2,"24/04/2020, 02/06/2023"
4,NT7105,NORTHUMBERLAND,2,"24/04/2020, 02/06/2023"
...,...,...,...,...
37435,TV5998,SOUTH DOWNS,5,"06/06/2013, 09/08/2016, 01/04/2019, 17/04/2022..."
37436,TV5999,SOUTH DOWNS,5,"06/06/2013, 09/08/2016, 01/04/2019, 17/04/2022..."
37437,TV6096,SOUTH DOWNS,5,"06/06/2013, 09/08/2016, 01/04/2019, 17/04/2022..."
37438,TV6097,SOUTH DOWNS,5,"06/06/2013, 09/08/2016, 01/04/2019, 17/04/2022..."


In [21]:
tile_dates.to_csv("../data/roi_tile_dates.csv", index=False)